# Spatial-scramble control
Is the causal patch effect **spatial**? For each seed we inject the donor's early up0 self-attention two ways: **intact**, and **spatially scrambled** (tokens randomly permuted -> same per-token channel content, destroyed layout). If intact shifts the count but scrambled does not, the effect depends on the **spatial arrangement**, not channel statistics.

**Runtime:** GPU (~15 min).

In [ ]:
import os
if not os.path.exists('src'):
    !git clone https://github.com/serinaqin/T2I-Count-Anomaly.git
    %cd T2I-Count-Anomaly
else:
    !git pull
!pip install -q -r requirements.txt
!pip install -q pytest groundingdino-py

In [ ]:
import sys; sys.path.insert(0, '.')
import numpy as np, pandas as pd, os, yaml
import matplotlib.pyplot as plt
from src.prompts import build_prompt
from src.pipeline import (load_sdxl, generate, catalog_attention_sites,
                          select_probe_sites, generate_and_capture,
                          raw_reducer, generate_with_patch, spatial_scramble)
from src.detector import Detector
from src.scoring import count_from_detections
from src.analysis import bootstrap_ci, sign_flip_pvalue
from src.config import load_config

In [ ]:
cfg = load_config('configs/exp_scramble.yaml')
raw = yaml.safe_load(open('configs/exp_scramble.yaml'))
psteps = raw['patch_steps']; pairs = raw['pairs']
block, attn = raw['patch_block'], raw['patch_attn']
pipe = load_sdxl(); det = Detector()
sites = [s for s in select_probe_sites(catalog_attention_sites(pipe.unet))
         if block in s and s.endswith(attn)]
def cnt(img, obj):
    return count_from_detections(det.detect(img, [obj]), obj, cfg.score_threshold)
def scramble_map(snaps, seed):
    return {st: {s: spatial_scramble(v, seed) for s, v in d.items()} for st, d in snaps.items()}

In [ ]:
rows = []
for obj in cfg.objects:
    for src, dnr in pairs:
        direction = 'up' if src < dnr else 'down'
        sp, dp = build_prompt(src, obj), build_prompt(dnr, obj)
        for seed in cfg.seeds:
            _, snaps = generate_and_capture(pipe, dp, seed, sites, psteps,
                                            cfg.num_inference_steps, reducer=raw_reducer)
            cb = cnt(generate(pipe, sp, seed, cfg.num_inference_steps), obj)
            cp_intact = cnt(generate_with_patch(pipe, sp, seed, snaps, cfg.num_inference_steps), obj)
            cp_scram = cnt(generate_with_patch(pipe, sp, seed, scramble_map(snaps, seed),
                                               cfg.num_inference_steps), obj)
            f = 1.0 if direction == 'up' else -1.0
            rows.append({'obj': obj, 'direction': direction, 'seed': seed,
                         'intact': f * (cp_intact - cb), 'scrambled': f * (cp_scram - cb)})
        print(obj, direction, 'done')
df = pd.DataFrame(rows)
os.makedirs('results', exist_ok=True)
df.to_csv('results/exp_scramble.csv', index=False)
df.head()

In [ ]:
# Donor-directed shift: intact vs spatially-scrambled donor.
for cond in ['intact', 'scrambled']:
    x = df[cond].to_numpy(float); lo, hi = bootstrap_ci(x)
    print(f'{cond:>10}: mean {x.mean():+.2f}  95% CI [{lo:+.2f}, {hi:+.2f}]  p={sign_flip_pvalue(x):.3f}')
fig, ax = plt.subplots(figsize=(5.5, 4))
for i, cond in enumerate(['intact', 'scrambled']):
    x = df[cond].to_numpy(float); lo, hi = bootstrap_ci(x)
    ax.errorbar(x.mean(), i, xerr=[[x.mean() - lo], [hi - x.mean()]], fmt='o', capsize=4,
                color='C0' if cond == 'intact' else 'C3')
ax.axvline(0, color='r', ls=':'); ax.set_yticks([0, 1]); ax.set_yticklabels(['intact', 'scrambled'])
ax.set_xlabel('donor-directed count shift'); ax.set_title('Does the patch need spatial arrangement?')
plt.tight_layout(); plt.savefig('results/exp_scramble.png', dpi=110, bbox_inches='tight'); plt.show()

## How to read this
- **intact shift clearly > 0, scrambled shift ~0** = the effect requires the donor's **spatial arrangement** -> it is a spatial layout intervention, not a channel-statistics / activation-magnitude one. Strong support for the spatial object-layout account (and consistent with why uniform steering failed).
- **scrambled shift ~ intact** = the effect is carried by channel/statistical content, not layout -> temper the 'spatial' claim.
- **both ~0** = weak effect at this n; lean on the scaled replication.